# CO₂ Blowdown with HEM Choke and Geothermal Initialization (Stage 2)

The headline configuration of the HyperCO2 study: single-phase gaseous CO₂
blowdown of the LSU research well (7-5/8" bore, 2,000 ft, 5 MPa wellhead,
12.7 mm choke, C_d = 0.84) with the two stage-2 measures that keep the
wellbore single-phase:

1. **HEM choke** — flashing is permitted *inside* the choke only, via a
   homogeneous-equilibrium critical-flow model (`G = ρ(p_t,s_0)·√(2[h_0−h])`
   maximized over throat pressure); the wellbore equations remain
   single-phase, with a saturation monitor that halts the run on any
   in-wellbore dome entry.
2. **Geothermal summer initialization** — `T(ζ,0) = T_s + Γζ` with
   `T_s = 305.4 K` (90 °F) and `Γ = 25.5 K/km`, hydrostatic pressure
   integrated with the real EOS, and the formation initialized from the
   same profile (zero initial wall flux).

This notebook runs a **reduced-resolution** version (n = 100 cells,
t_end = 1200 s, ≈2–5 min in Colab). The production campaign
(n = 300, 3600–4200 s) is `scripts/run_co2_campaign2.py`; its headline
numbers are quoted at the end for comparison.

In [ ]:
# Install from GitHub (Colab) -- or use a local clone
try:
    import hyperbolic1d, CoolProp
except ImportError:
    %pip install -q CoolProp "hyperbolic1d[plot] @ git+https://github.com/JaderBarbosaLSU/hyperco2-wellbore"
import numpy as np, matplotlib.pyplot as plt
from hyperbolic1d.realfluid import SpanWagnerCO2
from hyperbolic1d.wellbore import WellboreModel, WellGeometry, ChokeValve
from hyperbolic1d.thermal import (FormationCoupledWall, Geothermal,
                                  lsu_co2_well_stack)
from hyperbolic1d.postprocess import FT, lsu_fiber_sensors

## 1. The HEM choke in isolation

Before the transient: evaluate the choke flux at the initial wellhead
state. At 5 MPa / 300 K the isentrope through the stagnation state
crosses the vapor dome inside the expansion, so the single-phase
(dome-clamped) bound and the HEM flux differ — the HEM throat is
two-phase with quality x_t ≈ 0.97.

In [ ]:
eos = SpanWagnerCO2()                       # HEM choke on by default
G_hem, choked = eos.orifice_mass_flux(5.0e6, 300.0, 101325.0)
print(f"HEM:          G = {G_hem:8.1f} kg/m2s, choked={choked}, "
      f"flashing={eos.last_choke_flashing}, "
      f"x_t={eos.last_choke_throat_quality}")
eos_sp = SpanWagnerCO2(hem_choke=False)     # single-phase, dome-clamped
G_sp, _ = eos_sp.orifice_mass_flux(5.0e6, 300.0, 101325.0)
print(f"single-phase: G = {G_sp:8.1f} kg/m2s  (HEM is "
      f"{100*(G_hem/G_sp-1):+.1f}%)")

## 2. Transient blowdown: conjugate wall + geothermal IC

Reduced resolution for notebook runtimes. `two_phase_entry` stays `None`
for this protocol — the geothermal column keeps a comfortable margin to
the dome (production minimum: 14.2 K at t ≈ 809 s near the bottom).

In [ ]:
GEO = Geothermal(T_surface=305.4, gradient=0.0255)   # 90 F, 1.4 F/100 ft
r1, *_ = lsu_co2_well_stack()
DEPTHS_FT = (50, 500, 1000, 1500, 1950)

m = WellboreModel(geometry=WellGeometry(depth=609.6, diameter=2*r1),
                  choke=ChokeValve(), eos=SpanWagnerCO2(),
                  gravity=True, friction=True, n_cells=100,
                  wall_model=FormationCoupledWall(geothermal=GEO))
r = m.solve(5.0e6, 300.0, 1200.0, init="hydrostatic", geothermal=GEO,
            n_snapshots=121, hist_stride=50,
            sensors=lsu_fiber_sensors(DEPTHS_FT))
print(f"{r.n_steps} steps, {r.wall_time:.0f} s wall;  "
      f"budgets: mass {r.mass_balance_error:.1e}, "
      f"energy {r.energy_balance_error:.1e}")
print("two-phase entry:", r.two_phase_entry)
print(f"min saturation margin: {min(r.sat_margin_min):.2f} K;  "
      f"choke flashing fraction: {np.mean(r.choke_flashing):.2f}")

## 3. Wellhead histories and saturation margin

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 6.5))
tm = r.t_hist/60
axes[0,0].plot(tm, r.p_wellhead/1e6); axes[0,0].set_ylabel("$p_{wh}$ [MPa]")
axes[0,1].plot(tm, r.T_wellhead);     axes[0,1].set_ylabel("$T_{wh}$ [K]")
axes[1,0].plot(tm, r.mdot);           axes[1,0].set_ylabel(r"$\dot m$ [kg/s]")
axes[1,1].plot(tm, r.sat_margin_min, label="min over column")
axes[1,1].axhline(0, color="k", lw=0.7)
axes[1,1].set_ylabel(r"$\min_\zeta (T - T_{sat})$ [K]"); axes[1,1].legend()
for ax in axes.flat: ax.set_xlabel("$t$ [min]"); ax.grid(alpha=0.3)
fig.suptitle("CO$_2$ blowdown, geothermal summer IC, HEM choke, "
             "conjugate wall (n=100, 1200 s)")
fig.tight_layout()

## 4. Profiles vs depth ζ and fiber-sensor temperatures

In [ ]:
zeta_ft = (609.6 - r.z)/FT
fig, axes = plt.subplots(1, 3, figsize=(12, 4.6), sharey=True)
ks = [np.argmin(abs(r.snap_times - t)) for t in (0, 120, 600, 1200)]
for k in ks:
    axes[0].plot(r.p[k]/1e6, zeta_ft, label=f"{r.snap_times[k]:.0f} s")
    axes[1].plot(r.T[k], zeta_ft)
    axes[2].plot(r.u[k], zeta_ft)
axes[0].set_xlabel("$p$ [MPa]"); axes[1].set_xlabel("$T$ [K]")
axes[2].set_xlabel("$u$ [m/s]")
axes[0].set_ylabel(r"depth $\zeta$ [ft]"); axes[0].set_ylim(2000, 0)
axes[0].legend(title="$t$")
for ax in axes: ax.grid(alpha=0.3)
fig.tight_layout()

fig, ax = plt.subplots(figsize=(8, 4))
meta = r.sensor_meta
for name, Tts in r.sensor_data.items():
    ls = "-" if meta[name]["location"] == "fluid" else "--"
    ax.plot(r.sensor_t/60, Tts, ls,
            label=f"{meta[name]['location']} {meta[name]['depth']/FT:.0f} ft")
ax.set_xlabel("$t$ [min]"); ax.set_ylabel("sensor $T$ [K]")
ax.grid(alpha=0.3); ax.legend(fontsize=7, ncol=2)
ax.set_title("Fiber-optic sensors (solid: fluid; dashed: 13-3/8\" "
             "external fiber)")
fig.tight_layout()

## 5. Comparison with the production campaign

Production results (n = 300, t_end = 3600 s, `run_co2_campaign2.py`;
checkpoints and JSON in `results/co2_stage2/`):

| quantity | production value |
|---|---|
| two-phase entry | none (single-phase throughout) |
| min saturation margin | 14.2 K at t = 809 s, near bottomhole |
| choke flashing | never (geothermal case); 9.6 min for uniform-300 K |
| wellhead T minimum | ≥ 299.5 K |
| mass/energy budget errors | 1e-16 – 1e-12 |

The reduced-resolution run reproduces these qualitatively; first-order
differences are resolution (choke flux sees the coarser boundary cell)
and the shorter window. The adiabatic counterpart (`wall_model=None`)
enters the dome at t ≈ 503 s at the wellhead — run it by deleting the
`wall_model` argument and watch `two_phase_entry`.